# 公開用注記（原実験Notebook）

このNotebookは開発時のコードを記録として公開したものです。実験ロジックは変更していません。

**実行上の依存関係：** 前段の12D初期モデルとGenesisロールアウトデータが必要です。中間データは公開されていません。

公開版での一連の再実行は確認していません。実行時は参照パスや必要ファイルを確認してください。


# Phase47：Genesis実測データによる12D上位PPO更新
Phase46で収集した**実測行動・old log probability・Critic value・報酬・次状態**を使い、追跡/回避それぞれ1回のPPO更新を行います。CPUのみで更新可能。Phase42原本は保持し、更新モデルはDriveの別フォルダに保存します。
**注意**：学習後のGenesis評価は別Phase。姿勢安定性の改善はこのNotebookだけでは証明できません。

In [ ]:
# Phase47：Genesis実測ロールアウトから12D上位PPOを1回更新（CPU）
# 既存チェックポイントは読み込み専用。更新済みモデルは新しいフォルダへ保存。
from google.colab import drive, files
from pathlib import Path
from torch.distributions import Normal
import torch, torch.nn as nn, numpy as np, pandas as pd
import io, zipfile, json, hashlib, uuid, time
from IPython.display import display
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
torch.set_num_threads(2)
torch.manual_seed(47)
root=Path("/content/drive/MyDrive/PhysicalAI_Go2_Pursuit")
folders=sorted((root/"training/upper12_transfer").glob("phase42_*"))
folder=next((p for p in folders if all((p/f"upper_{role}_12d_initialized.pt").is_file() for role in ("pursuit","evasion"))),None)
assert folder is not None,"Phase42の12D初期モデルが見つかりません"
paths={role:folder/f"upper_{role}_12d_initialized.pt" for role in ("pursuit","evasion")}
def sha256(p):return hashlib.sha256(p.read_bytes()).hexdigest()
original={role:sha256(p) for role,p in paths.items()}
print("初期モデル",folder,"SHA256",original)
uploaded=files.upload()
assert len(uploaded)==1,"Phase46の結果ZIPを1つアップロードしてください"
zip_name,zip_bytes=next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
 candidates=[n for n in z.namelist() if n.endswith("/ppo_training_rollout.csv")]
 assert len(candidates)==1,"ppo_training_rollout.csv が見つかりません"
 df=pd.read_csv(z.open(candidates[0]))
assert len(df)==160 and set(df.robot)=={"A","B"}
assert set(df.role)=={"pursuer","evader"}
assert (df.groupby("robot").size()==80).all()
assert np.allclose(df.time_after_s-df.time_before_s,.02,atol=1e-6)
assert not df.isna().any().any()
assert df[["done_fall","done_role_switch"]].sum().sum()==0
assert df.truncated.sum()==2
obs_cols=[f"obs_{i}" for i in range(12)]
next_cols=[f"next_obs_{i}" for i in range(12)]
action_cols=[f"action_latent_{i}" for i in range(2)]
assert np.isfinite(df[obs_cols+next_cols+action_cols+["old_log_prob","old_value","reward"]].to_numpy()).all()
print("入力チェックPASS",len(df),"遷移")
class ActorCritic(nn.Module):
 def __init__(self):
  super().__init__()
  self.actor=nn.Sequential(nn.Linear(12,128),nn.Tanh(),nn.Linear(128,128),nn.Tanh(),nn.Linear(128,2))
  self.critic=nn.Sequential(nn.Linear(12,128),nn.Tanh(),nn.Linear(128,128),nn.Tanh(),nn.Linear(128,1))
  self.log_std=nn.Parameter(torch.tensor([-.5,-.5]))
 def dist(self,o):
  mu=self.actor(o)
  return Normal(mu,self.log_std.exp().expand_as(mu))
def load(role):
 ck=torch.load(paths[role],map_location="cpu",weights_only=False)
 assert ck["config"]["trained_on_12d"] is False
 model=ActorCritic()
 model.load_state_dict(ck["actor_critic"],strict=True)
 return model,ck
# 収集時の方策と現在のモデルが一致することを確認。ずれていたら学習しない。
buffers={}
models={}
checks=[]
gamma=.99
gae_lambda=.95
for role,robot in (("pursuit","A"),("evasion","B")):
 model,ck=load(role)
 sub=df[(df.robot==robot)&(df.role==("pursuer" if role=="pursuit" else "evader"))].sort_values("step").reset_index(drop=True)
 assert len(sub)==80 and sub.step.tolist()==list(range(1,81))
 obs=torch.tensor(sub[obs_cols].to_numpy(),dtype=torch.float32)
 nxt=torch.tensor(sub[next_cols].to_numpy(),dtype=torch.float32)
 action=torch.tensor(sub[action_cols].to_numpy(),dtype=torch.float32)
 old_logp=torch.tensor(sub.old_log_prob.to_numpy(),dtype=torch.float32)
 old_v=torch.tensor(sub.old_value.to_numpy(),dtype=torch.float32)
 reward=torch.tensor(sub.reward.to_numpy(),dtype=torch.float32)
 terminal=torch.tensor((sub.done_fall|sub.done_role_switch).to_numpy(),dtype=torch.float32)
 with torch.no_grad():
  check_logp=model.dist(obs).log_prob(action).sum(-1)
  check_v=model.critic(obs).squeeze(-1)
  logp_err=float((check_logp-old_logp).abs().max())
  value_err=float((check_v-old_v).abs().max())
  print(role,"old_logp最大差",logp_err,"old_value最大差",value_err)
  assert logp_err<1e-4 and value_err<1e-4,"Phase46収集時とチェックポイントが一致しません"
  next_v=model.critic(nxt).squeeze(-1)
  delta=reward+gamma*(1-terminal)*next_v-old_v
  adv=torch.zeros_like(delta)
  running=torch.tensor(0.)
  for i in reversed(range(len(delta))):
   running=delta[i]+gamma*gae_lambda*(1-terminal[i])*running
   adv[i]=running
  targets=adv+old_v
  # GAEは最後のtruncationでは次状態価値をbootstrapするが、後続の軌跡は存在しない
  # 次の役割/エピソードへの接続はしない
  norm_adv=(adv-adv.mean())/(adv.std(unbiased=False)+1e-8)
 buffers[role]=(obs,action,old_logp,old_v,targets,norm_adv)
 models[role]=(model,ck)
 checks.append(dict(role=role,logp_max_abs_error=logp_err,value_max_abs_error=value_err,
                    advantage_mean=float(adv.mean()),advantage_std=float(adv.std(unbiased=False)),
                    target_mean=float(targets.mean())))
display(pd.DataFrame(checks))
# まず1 epochだけの実測PPO更新。値関数の大幅な変化を避けるため小さい係数を使う。
updates=[]
out_dir=root/"training/upper12_genesis"/("phase47_"+uuid.uuid4().hex[:8])
out_dir.mkdir(parents=True,exist_ok=False)
for role in ("pursuit","evasion"):
 model,ck=models[role]
 obs,action,old_logp,old_v,targets,adv=buffers[role]
 optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)
 before=model.actor[0].weight[:,8:].detach().clone()
 model.train()
 dist=model.dist(obs)
 logp=dist.log_prob(action).sum(-1)
 ratio=(logp-old_logp).exp()
 policy_loss=-torch.minimum(ratio*adv,torch.clamp(ratio,.8,1.2)*adv).mean()
 value_pred=model.critic(obs).squeeze(-1)
 value_loss=.5*(value_pred-targets).square().mean()
 entropy=dist.entropy().sum(-1).mean()
 loss=policy_loss+.05*value_loss-.001*entropy
 assert torch.isfinite(loss)
 optimizer.zero_grad();loss.backward()
 grad=float(torch.nn.utils.clip_grad_norm_(model.parameters(),1.0))
 assert np.isfinite(grad)
 added_grad=float(model.actor[0].weight.grad[:,8:].abs().max())
 assert added_grad>0,"姿勢4入力への勾配がありません"
 optimizer.step()
 added_delta=float((model.actor[0].weight[:,8:]-before).abs().max())
 assert added_delta>0,"姿勢4入力が更新されませんでした"
 # 元のcheckpointの他のメタデータは保持し、state_dictのみ更新
 updated=dict(ck)
 updated["actor_critic"]=model.state_dict()
 updated["phase47_metadata"]={"source":zip_name,"samples":len(obs),"learning_rate":1e-4,
    "ppo_epochs":1,"gamma":gamma,"gae_lambda":gae_lambda,"value_coef":.05,
    "experimental":True,"genesis_evaluation_done":False}
 target=out_dir/f"upper_{role}_12d_phase47.pt"
 torch.save(updated,target)
 updates.append(dict(role=role,samples=len(obs),loss=float(loss.detach()),
    policy_loss=float(policy_loss.detach()),value_loss=float(value_loss.detach()),
    entropy=float(entropy.detach()),grad_norm=grad,
    added_weight_grad_max=added_grad,added_weight_update_max=added_delta,
    checkpoint=str(target)))
for role,p in paths.items():assert sha256(p)==original[role],"原本の変更を検出"
print("1回のPPO更新完了。元モデルは不変。")
display(pd.DataFrame(updates))
manifest={"phase":47,"source_zip":zip_name,"source_sha256":hashlib.sha256(zip_bytes).hexdigest(),
 "phase46_genesis_rollout":True,"real_actions_and_log_probs":True,
 "ppo_update_count_per_role":1,"genesis_evaluation_after_training":False,
 "original_checkpoint_sha256":original,"output_directory":str(out_dir),
 "limitations":["80step×2の単一短時間軌跡","姿勢ペナルティが0のデータ",
 "学習後のGenesis安定性は未検証","役割交代データは今回含まれない"]}
results=Path("/content/phase47_genesis_ppo_update_results.zip")
with zipfile.ZipFile(results,"w",zipfile.ZIP_DEFLATED) as z:
 z.writestr("ppo_update_metrics.csv",pd.DataFrame(updates).to_csv(index=False))
 z.writestr("policy_consistency_checks.csv",pd.DataFrame(checks).to_csv(index=False))
 z.writestr("manifest.json",json.dumps(manifest,ensure_ascii=False,indent=2))
print("結果ZIP",results)
files.download(str(results))

